In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             confusion_matrix, classification_report, roc_curve, auc)
import tensorflow as tf
import os
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Verify TensorFlow
print("TensorFlow Version:", tf.__version__)
try:
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
    from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
    print("Keras imports successful!")
except ImportError as e:
    print("Keras import error:", e)
    print("Please reinstall TensorFlow: pip uninstall tensorflow -y; pip install tensorflow==2.18.0")
    raise

In [ ]:
# Set random seed for reproducibility
np.random.seed(42)
tf.random.set_seed(42)


In [ ]:
try:
    train_df = pd.read_csv('../Binary-Labeled-CICIDS/train.csv')
    val_df = pd.read_csv('../Binary-Labeled-CICIDS/val.csv')
    test_df = pd.read_csv('../Binary-Labeled-CICIDS/test.csv')
except FileNotFoundError as e:
    print("Error: Preprocessed data files not found. Ensure 'train.csv', 'val.csv', 'test.csv' exist.")
    raise


In [ ]:
# Separate features and labels
X_train = train_df.drop('binary_label', axis=1)
y_train = train_df['binary_label']
X_val = val_df.drop('binary_label', axis=1)
y_val = val_df['binary_label']
X_test = test_df.drop('binary_label', axis=1)
y_test = test_df['binary_label']

print("Loaded preprocessed data:")
print(f"Train shape: {X_train.shape}, Validation shape: {X_val.shape}, Test shape: {X_test.shape}")


In [ ]:
# Evaluation function
def evaluate_model(y_true, y_pred, y_prob, model_name):
    print(f"\n{model_name} Evaluation:")
    print("Accuracy:", accuracy_score(y_true, y_pred))
    print("Precision:", precision_score(y_true, y_pred))
    print("Recall:", recall_score(y_true, y_pred))
    print("F1-Score:", f1_score(y_true, y_pred))
    print("ROC-AUC:", roc_auc_score(y_true, y_prob))
    print("\nClassification Report:\n", classification_report(y_true, y_pred))
    
    # Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'{model_name} Confusion Matrix')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.savefig('shallow_dnn_confusion_matrix.png')
    plt.close()
    
    # ROC Curve
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    roc_auc = auc(fpr, tpr)
    plt.figure(figsize=(6, 4))
    plt.plot(fpr, tpr, label=f'ROC curve (AUC = {roc_auc:.2f})')
    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'{model_name} ROC Curve')
    plt.legend(loc="lower right")
    plt.savefig('shallow_dnn_roc_curve.png')
    plt.close()
    
    # Return metrics
    return {
        'Model': model_name,
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred),
        'Recall': recall_score(y_true, y_pred),
        'F1-Score': f1_score(y_true, y_pred),
        'ROC-AUC': roc_auc_score(y_true, y_prob)
    }


In [ ]:
# Define Shallow DNN
def create_shallow_dnn(input_dim):
    model = Sequential([
        Dense(64, input_dim=input_dim, activation='relu'),
        BatchNormalization(),
        Dropout(0.3),
        Dense(32, activation='relu'),
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

In [ ]:

# Train Shallow DNN
input_dim = X_train.shape[1]
shallow_dnn = create_shallow_dnn(input_dim)
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
checkpoint = ModelCheckpoint('shallow_dnn_best.h5', monitor='val_loss', save_best_only=True)

history = shallow_dnn.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=128,
    callbacks=[early_stopping, checkpoint],
    verbose=1
)


In [ ]:
# Save model
shallow_dnn.save('shallow_dnn_final.h5')
print("Shallow DNN model trained and saved.")

In [ ]:
# Evaluate on test set
shallow_pred = (shallow_dnn.predict(X_test, verbose=0) > 0.5).astype(int).flatten()
shallow_prob = shallow_dnn.predict(X_test, verbose=0).flatten()
shallow_metrics = evaluate_model(y_test, shallow_pred, shallow_prob, "Shallow DNN")


In [ ]:
# Plot training history
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Shallow DNN Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Shallow DNN Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.tight_layout()
plt.savefig('shallow_dnn_training_history.png')
plt.close()


In [ ]:
# Save metrics
metrics_df = pd.DataFrame([shallow_metrics]).set_index('Model')
metrics_df.to_csv('shallow_dnn_metrics.csv')
print("\nShallow DNN Metrics:")
print(metrics_df)